# Notebook 2: nested hyperparameter selection, and what it cannot buy you

Companion to `REM_Turku_handoff.ipynb`; read that first.

**The short version, so you can decide whether to spend GPU here at all:** we measured,
three independent ways, that config selection extracts nothing at this data scale
(113 awakenings):

1. A 14-config sweep reproduced library defaults to 0.0006 (0.6462 vs 0.6456).
2. The dev-split spread could not rank the configs (differences inside split noise).
3. Nested selection changed nothing on any of five arms across two architectures
   (deltas -0.020 to +0.046, all inside fold-to-fold sd 0.08-0.16), against a
   pre-registered prediction that it would LOWER the numbers. It did not, because there
   was no optimism to remove: **the 12 sampled configs span 0.39-0.63 on inner
   validation while the outer sd is of the same order.** Selection is choosing among
   configs separated by less than the noise.

The same pattern is appearing on 101-Nights body_action (inner winners 0.64-0.65
delivering 0.38-0.46 on held-out folds).


## Why nesting matters even when it changes nothing

A sweep that selects on one dev split and then reports accuracy from that same split
family produces a number whose optimism you cannot bound. Nesting (select INSIDE each
training fold, score ONCE on the held-out subject) makes the estimate honest. Here the
honest estimate equals the naive one, which is itself the finding: the selection signal
is below the noise floor, measured.

Our full runs used budget=12, 3 inner folds, 17 outer folds: 8-12 h per
(architecture, target) arm on an H100. The default below is budget=3 with one target so
the cell finishes on a free Colab GPU. Do not expect the number to move; the value is
the diagnostic printed at the end.


In [ ]:
import sys, numpy as np
sys.path.insert(0, "..")
from harness.baseline_remturku import grid_for, sample_configs, train_eval
# grid_for(n_channels) = the tuning_p10_v3 wandb sweep parameters, with the three
# binding constraints (F1*D <= n_channels; depthwise kernel swept; real sampling).
GRID = grid_for(24)
print({k: v[:4] for k, v in list(GRID.items())[:6]})


In [ ]:
def nested_evaluate(arch, target, budget=3, inner_folds=3, seed=0):
    """Per outer LOSO fold: sample `budget` configs from GRID, score each on inner
    splits of the training subjects (train_eval: her optimizer + three-phase OneCycleLR
    + early stopping), retrain the winner, score once on the held-out subject.
    Failed configs score NaN; if all fail the run stops (never argmax over zeros)."""
    Xf, y_awk, s_awk = load(target)
    Xe, ye, se = [], [], []
    for x, yy, ss in zip(Xf, y_awk, s_awk):
        Xe.append(x); ye += [int(yy)] * len(x); se += [ss] * len(x)
    Xe = np.concatenate(Xe)[..., None]; ye = np.array(ye); se = np.array(se)
    rng = np.random.default_rng(seed)
    out = {}
    for held in np.unique(se):
        tr_subj = [s for s in np.unique(se) if s != held]
        if len(np.unique(ye[se == held])) < 2:
            continue
        configs = sample_configs(GRID, budget, rng)
        scores = np.full(budget, np.nan)
        for ci, cfg in enumerate(configs):
            accs = []
            for k in range(inner_folds):
                va_s = tr_subj[k::inner_folds]
                tr_m = np.isin(se, [s for s in tr_subj if s not in va_s])
                va_m = np.isin(se, va_s)
                try:
                    preds, _ = train_eval(Xe[tr_m], ye[tr_m], Xe[va_m], ye[va_m],
                                          Xe[va_m], ye[va_m], arch=arch, cfg=cfg,
                                          seed=seed)
                    accs.append(bal(preds, ye[va_m]))
                except Exception as e:
                    print("config failed:", e); accs = None; break
            if accs:
                scores[ci] = float(np.mean(accs))
        if np.all(np.isnan(scores)):
            raise SystemExit("every config failed; fix before trusting anything")
        win = configs[int(np.nanargmax(scores))]
        tr_m, te_m = se != held, se == held
        preds, _ = train_eval(Xe[tr_m], ye[tr_m], Xe[te_m], ye[te_m],
                              Xe[te_m], ye[te_m], arch=arch, cfg=win, seed=seed)
        out[str(held)] = {"outer": bal(preds, ye[te_m]),
                          "inner_span": (float(np.nanmin(scores)), float(np.nanmax(scores)))}
        print(held, out[str(held)])
    spans = [v["inner_span"] for v in out.values()]
    outer = [v["outer"] for v in out.values()]
    print(f"DIAGNOSTIC: inner spans {spans[:3]}... vs outer sd {np.std(outer):.3f}")
    return out


## The diagnostic that costs nothing

Before spending GPU-hours, print, per outer fold: the min/max inner score across
configs (the span) and, at the end, the sd of outer scores across folds. If the span
sits inside the outer sd, selection cannot rank configs on this data and the tuned
number will equal the default number, as it did for us on five of five arms. That
one print is the honest answer to "did you tune it?".
